# Semantic Search Engine Interactive Activity
*AIPI 510: Data Sourcing for Analytics*

*Week 13*



---



**Build Your Own Semantic Search Engine**

> **Goal**: Create a semantic search app from
scratch and deploy it to Hugging Face Spaces.


1.   Pick a dataset
2.  Chunk + embed your data
3. Store your data in a Chroma vector db
4. Build a search app using Streamlit or Gradio
5. Publish on Hugging Face Spaces
6. Submit Link on Canvas



---

*This is a starter notebook. This is designed to give you a jumping off point only. *

Make sure to
`pip install chromadb gradio sentence-transformers`


In [7]:
import pandas as pd
import numpy as np
import os

In [3]:
import sys; print(sys.executable)

/Users/prafulchunchu/Desktop/Duke/AIPI510/week13/.venv/bin/python


In [6]:
# Load an embedding model (you can use any embedding model)
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

W1119 21:50:58.834000 5861 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
gym_df = pd.read_csv('/Users/prafulchunchu/Desktop/Duke/AIPI510/week13/data/megaGymDataset.csv')
gym_df.head()

,Unnamed: 0,Title,Desc,Type,BodyPart,Equipment,Level,Rating,RatingDesc
0,0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,0.0,NaN
1,1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
2,2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
3,3,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
4,4,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate,NaN,NaN


In [33]:
# Little bit of EDA before chunking strategy

print("Shape of data frame: " + str(gym_df.shape))
print(f"Nulls of data frame: \n {gym_df.isna().sum()}")
print(f"Data types of the data frame: \n {gym_df.dtypes}")
print(f"Checking for duplicates: \n {gym_df.duplicated().sum()}")
print(f"Checking for length of Description as model only reads 1000 characters long (256 tokens): \n {gym_df['Desc'].str.len().describe()}")

Shape of data frame: (2918, 10)
Nulls of data frame: 
 Unnamed: 0       0
Title            0
Desc          1550
Type             0
BodyPart         0
Equipment       32
Level            0
Rating        1887
RatingDesc    2056
text             0
dtype: int64
Data types of the data frame: 
 Unnamed: 0      int64
Title             str
Desc              str
Type              str
BodyPart          str
Equipment         str
Level             str
Rating        float64
RatingDesc        str
text              str
dtype: object
Checking for duplicates: 
 0
Checking for length of Description as model only reads 1000 characters long (256 tokens): 
 count    1368.000000
mean      322.334064
std       107.566139
min        57.000000
25%       251.000000
50%       318.500000
75%       394.000000
max       691.000000
Name: Desc, dtype: float64


In [25]:
print(gym_df['Type'].value_counts())
print(gym_df['BodyPart'].value_counts())
print(gym_df['Equipment'].value_counts())
print(gym_df['Level'].value_counts())

Type
Strength                 2545
Stretching                147
Plyometrics                97
Powerlifting               37
Cardio                     35
Olympic Weightlifting      35
Strongman                  22
Name: count, dtype: int64
BodyPart
Abdominals     662
Quadriceps     646
Shoulders      340
Chest          262
Biceps         168
Triceps        151
Lats           124
Hamstrings     121
Middle Back    118
Lower Back      97
Glutes          81
Calves          47
Forearms        31
Traps           24
Abductors       21
Adductors       17
Neck             8
Name: count, dtype: int64
Equipment
Body Only        1078
Dumbbell          516
Barbell           282
Other             254
Cable             226
Machine           175
Kettlebells       149
Bands             100
Medicine Ball      38
Exercise Ball      35
E-Z Curl Bar       22
Foam Roll          11
Name: count, dtype: int64
Level
Intermediate    2446
Beginner         459
Expert            13
Name: count, dtype: int64


In [29]:
cols = ['Title', 'BodyPart', 'Equipment', 'Type', 'Desc']

gym_df['text'] = gym_df[cols].apply(
    lambda row: '. '.join(row.dropna().astype(str)), axis=1
)

In [35]:
# Provide your own dataset here. Decide on a chunking strategy and implement here.
texts = gym_df['text']
print(len(texts))

print(texts[0])

texts = texts.tolist()
print(len(texts))

2918
Partner plank band row. Abdominals. Bands. Strength. The partner plank band row is an abdominal exercise where two partners perform single-arm planks while pulling on the opposite ends of an exercise band. This technique can be done for time or reps in any ab-focused workout.
2918


In [36]:
# Generate embeddings
embeddings = model.encode(texts)

In [40]:
embeddings.shape

(2918, 384)

In [12]:
# Store embeddings in ChromaDB
import chromadb
from chromadb.config import Settings
client = chromadb.PersistentClient(path='chroma_db', settings=Settings(anonymized_telemetry=False))
collection = client.get_or_create_collection("exercises")
collection.add(documents=texts, embeddings=embeddings.tolist(), ids=[str(i) for i in range(len(texts))])

NameError: name 'texts' is not defined

In [2]:
collection.count()

2918

In [9]:
# Define a search function
def semantic_search(query):
    query_embedding = model.encode([query])
    results = collection.query(query_embeddings=query_embedding.tolist(), n_results=5)
    return "\n\n".join(results['documents'][0])

In [10]:
# Simple Gradio interface
import gradio as gr
demo = gr.Interface(
    fn=semantic_search,
    inputs=gr.Textbox(label="Enter your search query"),
    outputs=gr.Textbox(label="Top Matches"),
    title="Semantic Search Engine",
    description="Search over your custom dataset using semantic similarity."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


### Next Steps:

- Save your chromadb
- Write code to load your chromadb
- Convert your loading, search function and app code to an app.py script
- Create a requirements.txt file
- Test your code!!
- Push your git repo to Hugging Face Spaces


___
Your file tree in your repository will look like this:

- chromadb/
  - chroma.sqlite3
- README.md
- requirements.txt
- app.py